In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import plotly
import plotly.express as px
import plotly.graph_objects as go

econ_df = pd.read_csv("economy-and-growth.csv")


env_df = pd.read_csv("environment.csv")


energy_df = pd.read_csv('energy-and-mining.csv')

meta_df = pd.read_csv('Metadata_Country_API_5_DS2_en_csv_v2_3060772.csv')

## Things to Address:

- add bullet points for some plots, some are missing
- Possibly try other metrics from dataset
- Writeup

In [ ]:
energy_df.columns

In [ ]:
econ_df.sort_values('Country Name')

In [ ]:
env_df.columns

In [ ]:
meta_df.columns

## Proposition  

"Developing countries contribute more to environmental damage than developed countries"

**Summary:** Developing countries are often associated with rising emissions due to rapid industrialization, population growth, and reliance on lower-cost, carbon-intensive energy sources. As these economies expand, their total CO₂ output increases, leading to the perception that they are the primary drivers of current environmental degradation.

**Why it matters:** This framing influences global climate policy by shifting responsibility toward developing nations, often justifying stricter emissions constraints on them while potentially overlooking historical contributions from already industrialized economies.

## Visual 1: CO2 Damage Trend

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------
# Replace with your actual file names if needed
# env_df = pd.read_csv("environment_data.csv")
# meta_df = pd.read_csv("metadata.csv")

# --------------------------------------------------
# 2. Keep needed metadata columns and merge
# --------------------------------------------------
meta_subset = meta_df[['Country Code', 'IncomeGroup']].copy()

merged = env_df.merge(meta_subset, on='Country Code', how='left')

# --------------------------------------------------
# 3. Classify countries into 3 groups
# --------------------------------------------------
def classify_income(group):
    if group == 'High income':
        return 'Developed'
    elif group in ['Upper middle income', 'Lower middle income']:
        return 'Developing'
    elif group == 'Low income':
        return 'Low income'
    else:
        return None

merged['DevelopmentStatus'] = merged['IncomeGroup'].apply(classify_income)

# --------------------------------------------------
# 4. Define columns
# --------------------------------------------------
rel_col = 'average_value_Adjusted savings: carbon dioxide damage (% of GNI)'
abs_col = 'average_value_Adjusted savings: carbon dioxide damage (current US$)'

# --------------------------------------------------
# 5. Clean relevant columns
# --------------------------------------------------
plot_cols = ['Year', 'DevelopmentStatus', rel_col, abs_col]
plot_df = merged[plot_cols].copy()

plot_df['Year'] = pd.to_numeric(plot_df['Year'], errors='coerce')
plot_df = plot_df.dropna(subset=['Year', 'DevelopmentStatus'])
plot_df['Year'] = plot_df['Year'].astype(int)

# --------------------------------------------------
# 6. Relative damage plot: CO2 damage (% of GNI)
# --------------------------------------------------
rel_df = plot_df[['Year', 'DevelopmentStatus', rel_col]].dropna()

rel_grouped = (
    rel_df.groupby(['Year', 'DevelopmentStatus'])[rel_col]
    .mean()
    .reset_index()
)

rel_pivot = rel_grouped.pivot(
    index='Year',
    columns='DevelopmentStatus',
    values=rel_col
)

plt.figure(figsize=(10, 6))

for group in ['Developed', 'Developing', 'Low income']:
    if group in rel_pivot.columns:
        plt.plot(rel_pivot.index, rel_pivot[group], marker='o', label=group)

plt.title('Average CO₂ Damage (% of GNI) Over Time by Income Group')
plt.xlabel('Year')
plt.ylabel('CO₂ Damage (% of GNI)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 7. Absolute damage plot: CO2 damage (current US$)
# --------------------------------------------------
abs_df = plot_df[['Year', 'DevelopmentStatus', abs_col]].dropna()

abs_grouped = (
    abs_df.groupby(['Year', 'DevelopmentStatus'])[abs_col]
    .mean()
    .reset_index()
)

abs_pivot = abs_grouped.pivot(
    index='Year',
    columns='DevelopmentStatus',
    values=abs_col
)

plt.figure(figsize=(10, 6))

for group in ['Developed', 'Developing', 'Low income']:
    if group in abs_pivot.columns:
        plt.plot(abs_pivot.index, abs_pivot[group], marker='o', label=group)

plt.title('Average CO₂ Damage (Current US$) Over Time by Income Group')
plt.xlabel('Year')
plt.ylabel('CO₂ Damage (Current US$)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Methods and rationale: separate out low income since they are not developing yet

Possible deceptions: narrow time frame to support proposition: 1970-2000 for first plot, 2010-2020 for second plot

## Alt Visual 1: Rate of Change of CO2 Damage Trend

In [ ]:
import matplotlib.pyplot as plt

pivot = grouped.pivot(index='Year', columns='DevelopmentStatus', values=co2_col)

plt.figure(figsize=(10, 6))

for group in ['Developed', 'Developing', 'Low income']:
    if group in pivot.columns:
        y = pivot[group].dropna()
        x = y.index

        plt.plot(x, y, marker='o', label=group)

        # add trend line
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        plt.plot(x, p(x), linestyle='--')

plt.title('Trend in CO₂ Damage (% of GNI) Over Time')
plt.xlabel('Year')
plt.ylabel('CO₂ Damage (% of GNI)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Method and rationale: the rate of increase of CO2 damage is much greater for developing than developed.

Same possible deception.

In [ ]:
# Bar graph: Natural Resource Rents by Income Group

# Metric
metric_col = 'average_value_Total natural resources rents (% of GDP)'

# Merge environment_df with meta_df to get IncomeGroup
env_with_income = environment_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by IncomeGroup, take mean
income_rents = env_with_income.groupby('IncomeGroup')[metric_col].mean().reset_index()

# Filter to rows with data
income_rents = income_rents.dropna()

# Sort by income level (low to high)
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_rents['IncomeGroup'] = pd.Categorical(income_rents['IncomeGroup'], categories=income_order, ordered=True)
income_rents = income_rents.sort_values('IncomeGroup')

# Create bar chart
fig = px.bar(
    income_rents,
    x='IncomeGroup',
    y=metric_col,
    color='IncomeGroup',
    color_discrete_sequence=['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4'],
    title='Natural Resource Rents as % of GDP by Income Group',
    labels={metric_col: 'Natural Resource Rents (% of GDP)', 'IncomeGroup': 'Income Group'}
)

fig.update_layout(template="plotly_white", showlegend=False)
fig.show()

print(f"Average Natural Resource Rents by Income Group:")
for _, row in income_rents.iterrows():
    print(f"  {row['IncomeGroup']}: {row[metric_col]:.2f}%")

### Counter-Argument of Proposition
**Counter-Proposition:** "Developed countries actually contribute more to environmental damage"

**Summary:** Developed countries account for a disproportionately large share of cumulative CO₂ emissions due to over a century of industrial activity. Even today, they maintain higher per-capita emissions and consumption levels, and many have effectively outsourced carbon-intensive production to developing countries while still driving global demand.

**Why it matters:** This perspective emphasizes historical accountability and consumption-based emissions, which are critical for equitable climate agreements. It supports the argument that developed nations should bear greater responsibility in mitigation efforts, technology transfer, and climate financing.

## Visual 1: Total CO2 Damage Trend

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------
# Uncomment and edit if needed
# env_df = pd.read_csv("environment_data.csv")
# meta_df = pd.read_csv("metadata.csv")

# --------------------------------------------------
# 2. Merge metadata
# --------------------------------------------------
meta_subset = meta_df[['Country Code', 'IncomeGroup']].copy()
merged = env_df.merge(meta_subset, on='Country Code', how='left')

# --------------------------------------------------
# 3. Classify into 3 groups
# --------------------------------------------------
def classify_income(group):
    if group == 'High income':
        return 'Developed'
    elif group in ['Upper middle income', 'Lower middle income']:
        return 'Developing'
    elif group == 'Low income':
        return 'Low income'
    else:
        return None

merged['DevelopmentStatus'] = merged['IncomeGroup'].apply(classify_income)

# --------------------------------------------------
# 4. Define absolute CO2 damage column
# --------------------------------------------------
abs_col = 'average_value_Adjusted savings: carbon dioxide damage (current US$)'

# --------------------------------------------------
# 5. Clean data
# --------------------------------------------------
counter_df = merged[['Year', 'DevelopmentStatus', abs_col]].copy()
counter_df['Year'] = pd.to_numeric(counter_df['Year'], errors='coerce')
counter_df[abs_col] = pd.to_numeric(counter_df[abs_col], errors='coerce')

counter_df = counter_df.dropna(subset=['Year', 'DevelopmentStatus', abs_col])
counter_df['Year'] = counter_df['Year'].astype(int)

# --------------------------------------------------
# 6. Plot 1: Total CO2 damage over time (SUM, best for counter-argument)
# --------------------------------------------------
yearly_total = (
    counter_df
    .groupby(['Year', 'DevelopmentStatus'])[abs_col]
    .sum()
    .reset_index()
)

yearly_total_pivot = yearly_total.pivot(
    index='Year',
    columns='DevelopmentStatus',
    values=abs_col
)

plt.figure(figsize=(10, 6))

for group in ['Developed', 'Developing', 'Low income']:
    if group in yearly_total_pivot.columns:
        plt.plot(yearly_total_pivot.index, yearly_total_pivot[group], marker='o', label=group)

plt.title('Total CO₂ Damage (Current US$) Over Time by Income Group')
plt.xlabel('Year')
plt.ylabel('Total CO₂ Damage (Current US$)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Alt Visual 1: Energy Consumption (past 10 years)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --------------------------------------------------
# 1. Merge energy data with metadata
# --------------------------------------------------
meta_subset = meta_df[['Country Code', 'IncomeGroup']].copy()
meta_subset['IncomeGroup'] = meta_subset['IncomeGroup'].astype(str).str.strip()

energy_merged = energy_df.merge(meta_subset, on='Country Code', how='left')

# --------------------------------------------------
# 3. Classify income groups
# --------------------------------------------------
def classify_income(group):
    if pd.isna(group):
        return None

    group = str(group).strip()

    if group == 'High income':
        return 'Developed'
    elif group in ['Upper middle income', 'Lower middle income']:
        return 'Developing'
    elif group == 'Low income':
        return 'Low income'
    else:
        return None

energy_merged['DevelopmentStatus'] = energy_merged['IncomeGroup'].apply(classify_income)

# --------------------------------------------------
# 4. Choose metric
# --------------------------------------------------
energy_col = 'average_value_Electric power consumption (kWh per capita)'

# backup option:
# energy_col = 'average_value_Energy use (kg of oil equivalent per capita)'

# --------------------------------------------------
# 5. Clean data
# --------------------------------------------------
df = energy_merged[['Year', 'DevelopmentStatus', energy_col]].copy()

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df[energy_col] = pd.to_numeric(df[energy_col], errors='coerce')

df = df.dropna(subset=['Year', 'DevelopmentStatus', energy_col])
df['Year'] = df['Year'].astype(int)

# --------------------------------------------------
# 6. Use the last 10 available years
# --------------------------------------------------
recent_years = sorted(df['Year'].unique())[-10:]
recent_df = df[df['Year'].isin(recent_years)]

# --------------------------------------------------
# 7. Compute average per group over last 10 years
# --------------------------------------------------
group_avg = (
    recent_df.groupby('DevelopmentStatus')[energy_col]
    .mean()
    .reindex(['Developed', 'Developing', 'Low income'])
)


# --------------------------------------------------
# 8. Plot all groups
# --------------------------------------------------
all_groups = ['Developed', 'Developing', 'Low income']
plot_values = [group_avg[g] if g in group_avg.index else np.nan for g in all_groups]

plt.figure(figsize=(8, 6))
bars = plt.bar(
    all_groups,
    [0 if pd.isna(v) else v for v in plot_values]
)

plt.title(f'Average Electric Power Consumption per Capita ({recent_years[0]}–{recent_years[-1]})')
plt.xlabel('Income Group')
plt.ylabel('kWh per capita')
plt.grid(axis='y', alpha=0.3)

for i, v in enumerate(plot_values):
    if pd.isna(v):
        plt.text(i, 0, 'No data', ha='center', va='bottom')
    else:
        plt.text(i, v, f'{v:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# --- column ---
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'

# --- clean ---
df2 = energy_merged[['Year', 'DevelopmentStatus', fossil_col]].copy()
df2['Year'] = pd.to_numeric(df2['Year'], errors='coerce')
df2[fossil_col] = pd.to_numeric(df2[fossil_col], errors='coerce')
df2 = df2.dropna(subset=['Year', 'DevelopmentStatus', fossil_col])
df2['Year'] = df2['Year'].astype(int)

# --- last 10 years ---
recent_years = sorted(df2['Year'].unique())[-10:]
recent_df2 = df2[df2['Year'].isin(recent_years)]

group_avg2 = (
    recent_df2.groupby('DevelopmentStatus')[fossil_col]
    .mean()
    .reindex(['Developed', 'Developing', 'Low income'])
)

groups2 = group_avg2.index.tolist()
values2 = group_avg2.values

# --- plot ---
plt.figure(figsize=(8, 6))
bars = plt.bar(groups2, values2)

plt.title(f'Fossil Fuel Energy Consumption (% of Total) ({recent_years[0]}–{recent_years[-1]})')
plt.xlabel('Income Group')
plt.ylabel('% of total energy')
plt.grid(axis='y', alpha=0.3)

for bar in bars:
    h = bar.get_height()
    if np.isfinite(h):
        plt.text(bar.get_x() + bar.get_width()/2, h, f'{h:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Bar graph: Average Fossil Fuel vs Renewable Energy Consumption by IncomeGroup

# Metrics
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'
renewable_col = 'average_value_Renewable energy consumption (% of total final energy consumption)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Group by IncomeGroup only, take mean of both metrics
income_energy = energy_with_income.groupby('IncomeGroup').agg({
    fossil_col: 'mean',
    renewable_col: 'mean'
}).reset_index()

# Filter to rows with data
income_energy = income_energy.dropna()

# Reshape data for grouped bar chart
income_energy_melted = income_energy.melt(
    id_vars='IncomeGroup',
    value_vars=[fossil_col, renewable_col],
    var_name='Energy Type',
    value_name='Percentage'
)

# Create grouped bar chart
fig = px.bar(
    income_energy_melted,
    x='IncomeGroup',
    y='Percentage',
    color='Energy Type',
    color_discrete_map={fossil_col: '#d62728', renewable_col: '#2ca02c'},
    title='Average Fossil Fuel vs Renewable Energy Consumption by Income Group',
    labels={'Percentage': 'Average Consumption (%)'},
    barmode='group'
)

fig.update_xaxes(title_text='Income Group')
fig.update_layout(
    template='plotly_white',
    width=1000,
    height=600,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='center',
        x=0.5
    ),
    margin=dict(l=60, r=30, t=110, b=60)
)
fig.show()

print(f"Income Groups: {income_energy['IncomeGroup'].unique()}")

In [ ]:
# Line plot: Fossil and Renewable Energy - Bottom 50% vs Top 50% Income Groups
# Fossil = solid line, Renewable = dashed line

# Metrics
fossil_col = 'average_value_Fossil fuel energy consumption (% of total)'
renewable_col = 'average_value_Renewable energy consumption (% of total final energy consumption)'

# Merge energy_and_mining_df with meta_df to get IncomeGroup
energy_with_income = energy_and_mining_df.merge(
    meta_df[['Country Code', 'IncomeGroup']],
    left_on='Country Code',
    right_on='Country Code',
    how='left'
)

# Create income tier grouping: Bottom 50% vs Top 50%
def categorize_income_tier(income_group):
    if income_group in ['Low income', 'Lower middle income']:
        return 'Bottom 50%'
    elif income_group in ['Upper middle income', 'High income']:
        return 'Top 50%'
    else:
        return income_group

energy_with_income['IncomeTier'] = energy_with_income['IncomeGroup'].apply(categorize_income_tier)

# Group by Year and IncomeTier, take mean of both metrics
year_tier_energy = energy_with_income.groupby(['Year', 'IncomeTier']).agg({
    fossil_col: 'mean',
    renewable_col: 'mean'
}).reset_index()

# Filter to rows with data
year_tier_energy = year_tier_energy.dropna()

# Create figure
fig = go.Figure()

# Define colors for each tier
colors = {
    'Bottom 50%': '#d62728',
    'Top 50%': '#1f77b4'
}

# Add lines for each tier
for tier in ['Bottom 50%', 'Top 50%']:
    data = year_tier_energy[year_tier_energy['IncomeTier'] == tier]

    # Fossil fuel - solid line
    fig.add_trace(go.Scatter(
        x=data['Year'],
        y=data[fossil_col],
        name=f'{tier} (Fossil)',
        mode='lines',
        line=dict(color=colors.get(tier, '#000000'), width=3, dash='solid'),
        legendgroup=tier,
        hovertemplate='%{fullData.name}<br>Year: %{x}<br>Consumption: %{y:.2f}%<extra></extra>'
    ))

    # Renewable - dashed line
    fig.add_trace(go.Scatter(
        x=data['Year'],
        y=data[renewable_col],
        name=f'{tier} (Renewable)',
        mode='lines',
        line=dict(color=colors.get(tier, '#000000'), width=3, dash='dash'),
        legendgroup=tier,
        hovertemplate='%{fullData.name}<br>Year: %{x}<br>Consumption: %{y:.2f}%<extra></extra>'
    ))

fig.update_layout(
    title='Fossil (solid) vs Renewable (dashed) Energy: Bottom 50% vs Top 50% Income Groups',
    xaxis_title='Year',
    yaxis_title='Energy Consumption (%)',
    template='plotly_white',
    hovermode='x unified',
    height=600
)

fig.show()

print(f"Years covered: {year_tier_energy['Year'].min()} to {year_tier_energy['Year'].max()}")